# DCO phase-noise shape calculations

This code is designed for the DCO phase noise shaping.

It can support extracting from simulations, or deducing the paper one.

First: Initializations.

In [ ]:
# Libraries needed
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

freqs = np.logspace(4, 7.8, 500)
omega = 2j * np.pi * freqs

! mkdir -p outputs

# PDK of the simulation ("ics55" or "ihp-sg13g2"), or "paper" for the constants of the original paper
version = "ics55"

## Check the DCO Path Noise Model

A direct comparison between the transistor simulation and the model

In [ ]:
# Run the simulation if needed (ISF phase-noise analysis, see src/isf_phase_noise.py)
do_sim = version != "paper"
done_sim = False

sim_csv = f'../sim/outputs/{version}/test_dco_noise.csv'
if do_sim:
    if not os.path.exists(sim_csv):
        os.system(f"make -C ../sim PDK={version} test_dco_noise")
    done_sim = os.path.exists(sim_csv)

# Extract the simulation data if existed
if done_sim:
    # Columns: freq, L_dBc_Hz (total), L_white_dBc_Hz (thermal, 1/f^2), L_flicker_dBc_Hz (flicker, ~1/f^3)
    df = pd.read_csv(sim_csv)
    sim_dco_f = df['freq'].to_numpy()
    sim_dco_pn = df['L_dBc_Hz'].to_numpy()
    sim_dco_pn_white = df['L_white_dBc_Hz'].to_numpy()
    sim_dco_pn_flicker = df['L_flicker_dBc_Hz'].to_numpy()
    sim_dco_fdec = np.log10(sim_dco_f)

    # The simulation already separates the thermal and flicker parts, so fit each one
    # with the slope of the model (-20 and -30 dB/dec) in the frequencies of interest
    flicker_pseudo_m = -30
    thermal_pseudo_m = -20
    fit = (sim_dco_f >= freqs[0]) & (sim_dco_f <= freqs[-1])
    flicker_pseudo_b = np.mean(sim_dco_pn_flicker[fit] - flicker_pseudo_m * sim_dco_fdec[fit])
    thermal_pseudo_b = np.mean(sim_dco_pn_white[fit] - thermal_pseudo_m * sim_dco_fdec[fit])

    # Find the flicker corner in the line crossing
    fkr_cnr_dec = (thermal_pseudo_b - flicker_pseudo_b) / (flicker_pseudo_m - thermal_pseudo_m)
    fkr_cnr = np.power(10, fkr_cnr_dec)

    # Thermal noise factor: L_white = K_WA / f^2
    K_WA = np.power(10, thermal_pseudo_b/10)

    # Show the components of the simulation
    plt.figure(figsize=(6, 4))
    plt.semilogx(sim_dco_f, sim_dco_pn, color="#000000", label="DCO PN (sim)")
    plt.semilogx(sim_dco_f, sim_dco_pn_white, "--", color="#1f2468", label="Thermal part (sim)")
    plt.semilogx(sim_dco_f, sim_dco_pn_flicker, "--", color="#1f6829", label="Flicker part (sim)")
    plt.axvspan(freqs[0], freqs[-1], color="#EEEEEE", label="Fitted range")
    plt.title('DCO Phase Noise - From Simulations (ISF)')
    plt.ylabel('Magnitude (dBc/Hz)')
    plt.xlabel('Frequency (Hz)')
    plt.grid(True, which="both", ls="-")
    plt.legend(fontsize=8, loc='lower left')
    plt.tight_layout()
    plt.savefig("outputs/sim_DCO_pn_parts.pdf")
    plt.show()

else:
    # Extract the constants according to the original paper (figure 2)
    # NOTE: Those are highly manual. You are free to just write whatever you want
    flicker_pseudo_m = -30  # The 30 is just the -30db, and calculate b by calculate mean(yp - m * xp)
    thermal_pseudo_m = -20  # The 20 is just the -20db, and calculate b by calculate mean(yp - m * xp)
    sim_dco_f = freqs.copy()  # np.logspace(4, 8, 1000)
    sim_dco_fdec = np.log10(sim_dco_f)

    # Flicker point a = 110e3 [Hz], -78 [db]
    flicker_a_f = 110e3
    flicker_a_pn_db = -78
    
    flicker_pseudo_b = flicker_a_pn_db - flicker_pseudo_m * np.log10(flicker_a_f)

    # Thermal point a = 10e6 [Hz], -125 [db]
    thermal_a_f = 10e6
    thermal_a_pn_db = -125
    thermal_pseudo_b = thermal_a_pn_db - thermal_pseudo_m * np.log10(thermal_a_f)

    # Find the flicker corner in the line crossing
    fkr_cnr_dec = (thermal_pseudo_b - flicker_pseudo_b) / (flicker_pseudo_m - thermal_pseudo_m)
    fkr_cnr = np.power(10, fkr_cnr_dec)

    # Get the K_VA (Thermal noise factor) by using any random value in the thermal area
    # I think it can work also in the flicker area
    K_WA_eval_f = thermal_a_f
    K_WA_eval_pn = np.power(10, thermal_a_pn_db/10)
    # K_WA = K_WA_eval_pn * np.power(fkr_cnr, 2) / 2
    K_WA = K_WA_eval_pn / (1/np.power(K_WA_eval_f, 2) + fkr_cnr/np.power(K_WA_eval_f, 3))

Kfkr_WA = fkr_cnr * K_WA  # Get the flicker noise factor
n_psd_vcoa = Kfkr_WA / np.power(freqs, 3) + K_WA / np.power(freqs, 2)  # Derived VCO spectrum
print(f"fkr_cnr = {fkr_cnr}, K_WA = {K_WA}, Kfkr_WA = {Kfkr_WA}")

flicker_disp_y = sim_dco_fdec*flicker_pseudo_m + flicker_pseudo_b
thermal_disp_y = sim_dco_fdec*thermal_pseudo_m + thermal_pseudo_b

plt.figure(figsize=(6, 4))
if done_sim:
    plt.semilogx(sim_dco_f, sim_dco_pn, color="#000000", label=f"DCO PN (sim)")
plt.semilogx(freqs, 10 * np.log10(n_psd_vcoa), color="#ff7575", label="DCO PN (derived)")
plt.semilogx(sim_dco_f, flicker_disp_y, color="#1f6829", label="Flicker slope approx")
plt.semilogx(sim_dco_f, thermal_disp_y, color="#1f2468", label="Thermal slope approx")

plt.title('DCO phase noise - SIM vs MODEL')
plt.ylabel('Magnitude (dB)')
plt.xlabel('Frequency (Hz)')
plt.grid(True, which="both", ls="-")
plt.legend(fontsize=8, loc='lower left')
plt.tight_layout()
plt.savefig("outputs/dco_cmp.pdf")
plt.show()